In [ ]:
!pip uninstall -y protobuf
!pip install protobuf==3.20.3

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os
import re
import seaborn as sns
import torch
import transformers

from datasets import Dataset

from math import ceil

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
from sklearn.model_selection import StratifiedShuffleSplit

from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    EarlyStoppingCallback,
    TrainingArguments,
    Trainer,
    logging
)

from tqdm.auto import tqdm

(OBS) Code block added to delete the SQL status of the checkpoint (it has 7 GBs, so it ocuppies much space on the output). Add this code block at the beginning of the script, right after the imports.

In [ ]:
path = "/kaggle/working/state.db"

if os.path.exists(path):
    os.remove(path)
    print("state.db deleted")
else:
    print("state.db not found")


In [ ]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)

    macro_f1 = f1_score(labels, predictions, average="weighted")
    accuracy = accuracy_score(labels, predictions)
    precision = precision_score(labels, predictions, average="weighted")
    recall = recall_score(labels, predictions, average="weighted")

    return {
        "macro_f1": macro_f1,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall
    }

In [ ]:
def stratified_sample(
    df: pd.DataFrame,
    size: int
):
    splitter = StratifiedShuffleSplit(
        n_splits=1,
        train_size=size,
        random_state=42
    )

    for sample_idx, _ in splitter.split(df, df["label"]):
        return df.iloc[sample_idx].copy()

In [ ]:
base_path = "/kaggle/input/sem-eval-2026-task-13-subtask-b/Task_B"

training_path = base_path + "/train.parquet"
validation_path = base_path + "/validation.parquet"
test_sample_path = base_path + "/test_sample.parquet"
test_full_path = base_path + "/test.parquet"

training_df = pd.read_parquet(training_path)
validation_df = pd.read_parquet(validation_path)
test_sample_df = pd.read_parquet(test_sample_path)
test_full_df = pd.read_parquet(test_full_path)

test_df = pd.merge(test_sample_df, test_full_df, on="code", how="inner")

In [ ]:
pretrained_model = "microsoft/unixcoder-base"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(pretrained_model)

The functions below preprocess code samples and erase the comments.

In [ ]:
def clean_c_like_hard(code: str) -> str:
    if not isinstance(code, str):
        return ""
    lines = code.split('\n')
    cleaned_lines = []
    for line in lines:
        if any(x in line for x in ['//', '/*', '*/', '#']) or re.match(r'^\s*\*', line):
            continue
        cleaned_lines.append(line)
    code = '\n'.join(cleaned_lines)
    return re.sub(r'\s+', ' ', code).strip()

def clean_python(code: str) -> str:
    if not isinstance(code, str):
        return ""
    code = re.sub(r'"""[\s\S]*?"""', '', code, flags=re.MULTILINE)
    code = re.sub(r"'''[\s\S]*?'''", '', code, flags=re.MULTILINE)
    lines = []
    for line in code.split('\n'):
        if re.match(r'^\s*#', line):
            continue
        if '#' in line:
            line = line.split('#')[0]
        lines.append(line)
    code = '\n'.join(lines)
    return re.sub(r'\s+', ' ', code).strip()

def clean_php(code: str) -> str:
    if not isinstance(code, str):
        return ""
    lines = code.split('\n')
    cleaned_lines = []
    for line in lines:
        if any(x in line for x in ['//', '/*', '*/', '#']) or re.match(r'^\s*\*', line):
            continue
        cleaned_lines.append(line)
    code = '\n'.join(cleaned_lines)
    return re.sub(r'\s+', ' ', code).strip()

def clean_code(code: str, language: str) -> str:
    lang = language.lower() if isinstance(language, str) else ""
    if lang == "python":
        return clean_python(code)
    elif lang in ["c", "cpp", "c++", "csharp", "java", "go", "js", "javascript"]:
        return clean_c_like_hard(code)
    elif lang == "php":
        return clean_php(code)
    else:
        return code.strip()

In [ ]:
def preprocess_function(examples: pd.DataFrame):
    # cleaned_code = []
    # for code, language in zip(examples["code"], examples["language"]):
    #     cleaned = clean_code(code, language)
    #     cleaned_code.append(cleaned)

    return tokenizer(examples["code"], truncation=True, max_length=256)

(OBS) It is good practice to set the format to torch after you tokenize the datasets. Add the .set_format("torch") instructions right after you tokenized the datasets.

In [ ]:
training_dataset = Dataset.from_pandas(training_df)
validation_dataset = Dataset.from_pandas(validation_df)
test_dataset = Dataset.from_pandas(test_df)

training_tokenized_set = training_dataset.map(preprocess_function, batched=True)
validation_tokenized_set = validation_dataset.map(preprocess_function, batched=True)
test_tokenized_set = test_dataset.map(preprocess_function, batched=True)

training_tokenized_set.set_format("torch")
validation_tokenized_set.set_format("torch")
test_tokenized_set.set_format("torch")

In [ ]:
id2label = {
    0: "Human",
    1: "deepseek-ai/DeepSeek-V3-0324",
    2: "Qwen/Qwen2.5-Coder-7B-Instruct",
    3: "01-ai/Yi-Coder-9B-Chat",
    4: "bigcode/starcoder",
    5: "gemma-3n-e4b-it",
    6: "microsoft/phi-2",
    7: "meta-llama/Llama-3.3-70B-Instruct-Turbo",
    8: "ibm-granite/granite-3.2-2b-instruct",
    9: "mistralai/Devstral-Small-2505",
    10: "GPT-4o-mini"
}

label2id = {
    "Human": 0,
    "deepseek-ai/DeepSeek-V3-0324": 1,
    "Qwen/Qwen2.5-Coder-7B-Instruct": 2,
    "01-ai/Yi-Coder-9B-Chat": 3,
    "bigcode/starcoder": 4,
    "gemma-3n-e4b-it": 5,
    "microsoft/phi-2": 6,
    "meta-llama/Llama-3.3-70B-Instruct-Turbo": 7,
    "ibm-granite/granite-3.2-2b-instruct": 8,
    "mistralai/Devstral-Small-2505": 9,
    "GPT-4o-mini": 10
}

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    pretrained_model,
    num_labels=11,
    id2label=id2label,
    label2id=label2id
)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print("\nRunning on device:", device)

In [ ]:
logging.set_verbosity_info()

In [ ]:
class TrainingProgressCallback(transformers.TrainerCallback):
    def on_step_end(self, args, state, control, **kwargs):
        if state.is_local_process_zero:
            tqdm.write(f"Step {state.global_step}/{state.max_steps}")

In [ ]:
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

(OBS) In the TrainingArguments parameters, you have to set the following:
- output_dir="/kaggle/working/checkpoints" (this is the Kaggle notebook folder that stores the output files; you will need to have persistent data, to achieve that the checkpoints must be saved in this folder);
- num_train_epochs=3;
- load_best_model_at_end=True (for best results);
- eval_strategy="epoch";
- save_strategy="epoch";
- save_total_limit=2 (or 3, you set here the last number of checkpoints that remain saved);
- logging_dir="/kaggle/working/logs" (optional, stores log information);
- logging_steps=3000 (optional, add only if you put logging_dir too);

In [ ]:
training_args = TrainingArguments(
    output_dir="/kaggle/working/checkpoints",
    seed=42,
    learning_rate=3e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    metric_for_best_model="macro_f1",
    load_best_model_at_end=True,
    logging_dir="/kaggle/working/logs",
    logging_steps=3000,
    fp16=True,
    disable_tqdm=False,
    ddp_find_unused_parameters=False,
    dataloader_num_workers=0,
    no_cuda=False,
    report_to=[]
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=training_tokenized_set,
    eval_dataset=validation_tokenized_set,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2),
               TrainingProgressCallback()]
)

(OBS) The first execution will have just trainer.train(), because initially you won't have any checkpoints, you start from 0 with the training. After that, you need to replace this instruction with trainer.train(resume_from_checkpoint=True), such that it resumes the training from the last checkpoint.

In [ ]:
trainer.train(resume_from_checkpoint=True)

In [ ]:
test_results = trainer.evaluate(test_tokenized_set)
print("Test results:", test_results)

predictions = trainer.predict(test_tokenized_set)
logits = predictions.predictions
labels = predictions.label_ids

predicted_labels = np.argmax(logits, axis=1)

cm = confusion_matrix(labels, predicted_labels)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=False, cmap="Blues")
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()

output_df = pd.DataFrame({
    "ID": test_df["ID"],
    "label": predicted_labels
})

output_df.to_csv("test_predictions.csv", index=False)